# CardioSentry — FCMAE epoch-20 → supervised ConvNeXt-V2 Tiny fine-tuning

Your `.pt` is treated correctly here as a **self-supervised FCMAE encoder after
20 pretraining epochs**. It has not yet learned the four diagnostic labels and
does not need an existing classifier head.

This Colab creates a fresh four-logit head for **STEMI, AF, LVH, and NORMAL** and
fine-tunes the entire dense ConvNeXt-V2 Tiny model under the same supervised
protocol used by `baselines_v2`: `nat768`, printed-ID redaction, patient-disjoint
splits, AdamW, effective batch 32, cosine schedule, up to 20 epochs, early
stopping on validation macro AUPRC, validation-tuned thresholds, and one final
test evaluation.

## What this notebook runs

Exactly one supervised run is performed: your ECG-FCMAE epoch-20 encoder is
loaded, a new four-label classification head is attached, and the whole model
is fine-tuned and evaluated. No initial encoder or second control model is
loaded or trained. Improvement can be inspected against the saved
`baselines_v2/comparison.csv` when that file is available in Drive.

---
Before starting: **Runtime → Change runtime type → GPU**. L4 or A100 preferred.
Only load checkpoints you created or trust.


## Cell 1 — unpack the self-contained supervised harness

The Colab embeds the official dense ConvNeXt-V2 Tiny implementation plus the
`baselines_v2` dataset checks, metrics, checkpointing, predictions, and reports.
No MinkowskiEngine build is needed for supervised fine-tuning.


In [ ]:
import base64, io, tarfile, os
os.makedirs("/content/code", exist_ok=True)
_PKG = '...'
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode="r:gz").extractall("/content/code")
for _f in os.listdir("/content/code"):
    if _f.endswith(".sh"):
        os.chmod(f"/content/code/{_f}", 0o755)
os.environ["CS_DIM"] = "nat768"
print("unpacked:")
for _f in sorted(os.listdir("/content/code")): print("  ", _f)
!pip install -q "timm>=1.0.9"
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Cell 2 — mount Drive and locate your pretrained encoder

The notebook searches for `encoder_epoch_0020.pt`; if absent, Colab opens an
upload picker for your pretrained `.pt` file. This is the only model checkpoint
used by the notebook.

Checkpoint loading requires every one of the 196 encoder tensors. Only
`norm.weight`, `norm.bias`, `head.weight`, and `head.bias` may be absent and are
freshly initialized for supervised fine-tuning.


In [ ]:
import glob, hashlib, os, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/CardioSentry"
DIM = "nat768"
MICRO_BATCH = 8                 # use 4 if a T4 preflight reports OOM
os.environ["CS_DIM"] = DIM
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"

TAR = f"{DRIVE}/datasets/prototype_1152.tar"
RAW = "/content/prototype_1152"
DATA = RAW
CODES = f"{DRIVE}/codes/convnextv2_tiny_fcmae20_finetune_v1"
EXP = f"{DRIVE}/experiments/convnextv2_tiny_fcmae20_finetune_v1"
CACHE = f"{EXP}/data"
CKPTS = f"{EXP}/checkpoints"
RUNS = f"{DRIVE}/runs/convnextv2_tiny_fcmae20_finetune_v1"
for _p in (CODES, CACHE, CKPTS, RUNS): Path(_p).mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""): h.update(block)
    return h.hexdigest()

# Your encoder after 20 ECG-domain FCMAE pretraining epochs.
# Paste an exact Drive path here if discovery should not be used.
FCMAE_CHECKPOINT = ""
if not FCMAE_CHECKPOINT:
    candidates = []
    candidates += glob.glob(f"{DRIVE}/runs/fcmae_tiny_v1_768_seed42/full/encoders/encoder_epoch_0020.pt")
    candidates += glob.glob(f"{DRIVE}/runs/**/encoders/encoder_epoch_0020.pt", recursive=True)
    candidates += glob.glob(f"{CKPTS}/fcmae_ecg_epoch0020*.pt")
    candidates = sorted(set(candidates), key=os.path.getmtime, reverse=True)
    if candidates:
        FCMAE_CHECKPOINT = candidates[0]
        print("Using your ECG-FCMAE epoch-20 encoder:", FCMAE_CHECKPOINT)
        for p in candidates[1:6]: print("  other candidate:", p)
    else:
        from google.colab import files
        print("Choose the trusted FCMAE epoch-20 encoder .pt file.")
        uploaded = files.upload()
        pts = [name for name in uploaded if name.lower().endswith((".pt", ".pth"))]
        if len(pts) != 1:
            raise RuntimeError(f"Upload exactly one .pt/.pth; received {list(uploaded)}")
        dest = Path(CKPTS) / "fcmae_ecg_epoch0020.pt"
        shutil.copy2(pts[0], dest)
        FCMAE_CHECKPOINT = str(dest)
        print("Persisted your pretrained encoder to", FCMAE_CHECKPOINT)

assert Path(FCMAE_CHECKPOINT).is_file(), FCMAE_CHECKPOINT
print("pretrained checkpoint sha256", sha256(FCMAE_CHECKPOINT))

if not os.path.exists(f"{RAW}/index.csv"):
    !tar -xf "$TAR" -C /content/
!ls "$RAW"
import pandas as pd
_raw_index = pd.read_csv(f"{RAW}/index.csv")
if len(_raw_index) != 6670:
    raise RuntimeError(
        f"Expected the baselines_v2 prototype_1152 index with 6,670 image rows; "
        f"found {len(_raw_index):,}. Refusing a non-comparable dataset.")
print("Verified baselines_v2 prototype_1152 index: 6,670 rows; "
      "runtime manifest checks will exclude the two documented corrupt records")
print("\nDATA =", DATA, "\nRUNS =", RUNS, "\nEXP =", EXP)


## Cell 3 — inspect the supervised input geometry

Renders the same `nat768` full sheet with the printed identifier redacted. This
is the actual preprocessing path used for fine-tuning.


In [ ]:
%cd /content/code
!python dimensions.py | head -12
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
import sys; sys.path.insert(0, "/content/code")
import dimensions as dimlib
from aspect_study import render
df = pd.read_csv(f"{RAW}/index.csv")
df = df[~df.record.isin(dimlib.EXCLUDED_RECORDS)]
picks = [df[df.cls == c].iloc[0] for c in ("STEMI", "LVH", "AF", "NORMAL")]
Path(f"{EXP}/samples").mkdir(parents=True, exist_ok=True)
for row in picks:
    image = render(Image.open(f"{RAW}/{row['rel_path']}").convert("RGB"), DIM)
    image.save(f"{EXP}/samples/{row['cls']}_{row['record']}.png")
    display(Markdown(f"### {row['cls']} — {row['record']}")); display(image)


## Cell 4 — prepare and cache the redacted `nat768` corpus

The first session writes one prepared-data tar to `experiments/`; subsequent
sessions restore it. Training reads local `/content`, not mounted Drive.


In [ ]:
%cd /content/code
!./prepare_corpus.sh "$RAW" "/content/prototype_{DIM}" "$CACHE"
_prepared = f"/content/prototype_{DIM}"
if os.path.exists(os.path.join(_prepared, "index.csv")):
    DATA = _prepared
else:
    DATA = RAW
    print("Prepared cache unavailable; RAW remains valid but slower")
print("DATA is", DATA)


## Cell 5 — integrity and GPU preflight

Checks split leakage, geometry/redaction, model shape, effective batching, and a
real forward/backward at 768×1024 before loading the long runs.


In [ ]:
%cd /content/code
!python selftest.py --data "$DATA"
!python preflight.py --data "$DATA" --model fcmae_ecg_epoch0020


## Cell 6 — checkpoint-aware smoke test

The smoke run strictly loads all encoder tensors and trains/evaluates a few
batches. Its metrics are deliberately disposable.


In [ ]:
%cd /content/code
!python train.py --model fcmae_ecg_epoch0020 --init-checkpoint "$FCMAE_CHECKPOINT" --data "$DATA" --out /content/smoke_fcmae20 --micro-batch "$MICRO_BATCH" --smoke


## Cell 7 — supervised fine-tune your pretrained model

This is the long-running cell: one supervised run of up to 20 epochs. Every
epoch, checkpoint, metric, and prediction is written directly to Drive. An
interrupted run resumes automatically.


In [ ]:
%cd /content/code
!python train.py --model fcmae_ecg_epoch0020 --init-checkpoint "$FCMAE_CHECKPOINT" --data "$DATA" --out "$RUNS" --micro-batch "$MICRO_BATCH" --epochs 20
!python compare.py --runs "$RUNS" --csv "$RUNS/comparison.csv"


## Cell 8 — final test results


In [ ]:
import json, pandas as pd
RUN_NAME = "fcmae_ecg_epoch0020@nat768"
result = json.load(open(f"{RUNS}/{RUN_NAME}/result.json"))
print(open(f"{RUNS}/{RUN_NAME}/report.txt").read())


## Cell 9 — headline metrics

The table mirrors the headline fields exported by `baselines_v2`.


In [ ]:
comparison = pd.read_csv(f"{RUNS}/comparison.csv")
cols = ["model", "macro_auroc", "macro_auprc", "macro_sensitivity",
        "macro_specificity", "macro_precision", "macro_f1", "macro_fpr",
        "exact_match_accuracy", "params_total_m", "size_mb", "latency_ms_b1"]
display(comparison[[c for c in cols if c in comparison]].sort_values("macro_auprc", ascending=False).round(4))


## Cell 10 — per-label test metrics


In [ ]:
per_label = pd.DataFrame(result["test"]["per_label"]).T
display(per_label[[c for c in ("auroc", "auprc", "sensitivity",
    "specificity", "precision", "npv", "f1", "fpr", "fnr",
    "accuracy", "balanced_accuracy", "threshold", "support", "tp",
    "fp", "fn", "tn") if c in per_label]].round(4))


## Cell 11 — supervised fine-tuning curves


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
h = pd.read_csv(f"{RUNS}/{RUN_NAME}/history.csv")
ax[0].plot(h.epoch, h.train_loss, marker="o")
ax[1].plot(h.epoch, h.val_macro_auprc, marker="o")
ax[2].plot(h.epoch, h.val_macro_auroc, marker="o")
for a, title in zip(ax, ("train loss", "validation macro AUPRC", "validation macro AUROC")):
    a.set_title(title); a.set_xlabel("supervised fine-tuning epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()


## Cell 12 — source-confound check (mandatory)

All STEMI records come from one database. Report per-source metrics and spread
beside the headline metrics.


In [ ]:
ps = result["test_per_source"]
values = [x["macro_auroc"] for x in ps.values()]
print("  ".join(f"{s}={d['macro_auroc']:.3f}(n={d['n']})" for s, d in ps.items()))
print("spread", max(values) - min(values) if len(values) > 1 else float("nan"))


## Cell 13 — optional context beside the nine `baselines_v2` models

This places the single fine-tuned model beside prior baseline results when
their saved comparison file is available. It does not train another model.


In [ ]:
BASELINES_CSV = f"{DRIVE}/runs/baselines_v2/comparison.csv"
if Path(BASELINES_CSV).exists():
    baseline = pd.read_csv(BASELINES_CSV)
    shared = [c for c in ("model", "macro_auroc", "macro_auprc", "macro_f1",
                           "macro_sensitivity", "macro_specificity", "params_total_m",
                           "latency_ms_b1") if c in baseline and c in comparison]
    context = pd.concat([baseline[shared], comparison[shared]], ignore_index=True)
    context.to_csv(f"{RUNS}/baseline_context.csv", index=False)
    display(context.sort_values("macro_auprc", ascending=False).round(4))
else:
    print("No baselines_v2/comparison.csv found; this fine-tuning run is still complete")


## Cell 14 — reports and artifact checklist


In [ ]:
print(open(f"{RUNS}/{RUN_NAME}/report.txt").read())
print("Saved run-level artifacts:")
for p in sorted(Path(RUNS).iterdir()):
    if p.is_file(): print(p.name, p.stat().st_size)


## Persistent outputs

```text
MyDrive/CardioSentry/
├── codes/convnextv2_tiny_fcmae20_finetune_v1/
│   └── CardioSentry_ConvNeXtV2_Tiny_FCMAE20_Finetune_Colab.ipynb
├── experiments/convnextv2_tiny_fcmae20_finetune_v1/
│   ├── checkpoints/                    your uploaded FCMAE epoch-20 encoder
│   ├── data/                           prepared nat768 cache
│   └── samples/                        rendered preprocessing QA
└── runs/convnextv2_tiny_fcmae20_finetune_v1/
    ├── _SUMMARY.txt
    ├── comparison.csv / comparison.txt
    ├── baseline_context.csv             only if baselines_v2 exists
    └── fcmae_ecg_epoch0020@nat768/
        ├── STATUS.txt / train_log.txt / history.csv
        ├── metrics.csv / result.json / report.txt
        ├── config_snapshot.json
        ├── weights/best.pt
        └── predictions/{val,test}_predictions.csv
```

Final metrics include per-label and macro AUROC, AUPRC, sensitivity,
specificity, precision, NPV, F1, FPR, FNR, accuracy, balanced accuracy,
thresholds, support, TP/FP/FN/TN; micro metrics; exact-match and Hamming
accuracy; per-source metrics; parameters, model size, latency, and throughput.

This notebook fine-tunes and reports one pretrained model. Any improvement is
read from its metrics relative to the already-saved `baselines_v2` results; no
extra control encoder is downloaded or trained.
